# Taller SLAM Visual Simulado con OpenCV
**Visual Odometry** usando ORB + Essential Matrix sobre una secuencia sintética.

**Estudiante:** Juan David Cardenas Galvis  
**Fecha:** 2026-06-08

---
## Objetivo
Simular los principios de SLAM (Simultaneous Localization and Mapping) detectando puntos clave entre frames consecutivos,
estimando el movimiento relativo de la cámara y construyendo una trayectoria 2D aproximada.

## 0 · Importaciones y configuración

In [ ]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image, display
import os

MEDIA_DIR = os.path.join('..', 'media')
os.makedirs(MEDIA_DIR, exist_ok=True)

print(f'OpenCV  : {cv2.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'Media   : {os.path.abspath(MEDIA_DIR)}')

## 1 · Generación de escena sintética
Creamos 150 puntos 3D aleatorios y los proyectamos sobre frames de 480×480 px simulando una cámara que describe un arco horizontal con ligero avance en Z.

In [ ]:
def generate_synthetic_scene(n_points=120, img_size=480):
    rng = np.random.default_rng(42)
    pts = rng.uniform(-3, 3, (n_points, 3)).astype(np.float32)
    pts[:, 2] += 6.0
    return pts

def project_points(pts3d, R, t, K, img_size=480):
    pts_cam = (R @ pts3d.T).T + t.flatten()
    valid = pts_cam[:, 2] > 0.1
    pts_cam = pts_cam[valid]
    uvw = (K @ pts_cam.T).T
    uv = uvw[:, :2] / uvw[:, 2:3]
    in_frame = (
        (uv[:, 0] >= 0) & (uv[:, 0] < img_size) &
        (uv[:, 1] >= 0) & (uv[:, 1] < img_size)
    )
    return uv[in_frame].astype(np.float32), pts_cam[in_frame]

def render_frame(pts3d, R, t, K, img_size=480):
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    rng = np.random.default_rng(int(abs(t.flatten()[0]) * 1000) % 9999)
    noise = rng.integers(0, 18, img.shape, dtype=np.uint8)
    img = cv2.add(img, noise)
    uv, _ = project_points(pts3d, R, t, K, img_size)
    for p in uv:
        x, y = int(p[0]), int(p[1])
        color = (int(80+(x/img_size)*175), int(80+(y/img_size)*175), 200)
        cv2.circle(img, (x, y), 4, color, -1)
        cv2.circle(img, (x, y), 6, (255, 255, 255), 1)
    return img

def build_frame_sequence(n_frames=60, img_size=480):
    K = np.array([
        [img_size*0.8, 0,            img_size/2],
        [0,            img_size*0.8, img_size/2],
        [0,            0,            1         ],
    ], dtype=np.float64)
    pts3d = generate_synthetic_scene(150, img_size)
    frames, camera_poses = [], []
    for i in range(n_frames):
        angle = (i / n_frames) * 2 * np.pi * 0.6
        cx, cy, cz = 1.8*np.sin(angle), 0.5*np.sin(angle*2)*0.4, i*0.06
        Rw = cv2.Rodrigues(np.array([0.0, angle*0.25, 0.0]))[0]
        tw = np.array([[cx],[cy],[cz]], dtype=np.float64)
        R_cam = Rw.T
        t_cam = -Rw.T @ tw
        frames.append(render_frame(pts3d, R_cam, t_cam, K, img_size))
        camera_poses.append((tw.flatten(), Rw))
    return frames, camera_poses, K

frames, gt_poses, K = build_frame_sequence(n_frames=60, img_size=480)
print(f'Secuencia generada: {len(frames)} frames de {frames[0].shape}')

### Previsualización de 4 frames de la secuencia

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
indices = [0, 19, 39, 59]
for ax, idx in zip(axes, indices):
    ax.imshow(cv2.cvtColor(frames[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {idx}')
    ax.axis('off')
plt.suptitle('Muestra de frames sintéticos', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'frames_muestra.png'), dpi=110)
plt.show()
print('Imagen guardada: frames_muestra.png')

## 2 · Detección de puntos clave con ORB
ORB (Oriented FAST and Rotated BRIEF) es un detector rápido, invariante a rotación, con descriptores binarios que permiten emparejamiento eficiente con Hamming distance.

In [ ]:
orb = cv2.ORB_create(nfeatures=1000)

# Detectar en frame central
sample = frames[30]
gray   = cv2.cvtColor(sample, cv2.COLOR_BGR2GRAY)
kps, _ = orb.detectAndCompute(gray, None)

out = cv2.drawKeypoints(
    sample, kps, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

fig, ax = plt.subplots(figsize=(6, 6), dpi=110)
ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
ax.set_title(f'Puntos clave ORB — {len(kps)} keypoints (frame 30)')
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'keypoints_orb.png'), dpi=110)
plt.show()
print(f'Keypoints detectados: {len(kps)}')

## 3 · Emparejamiento de características entre frames
Usamos **BFMatcher** con norma Hamming (adecuada para descriptores binarios ORB) con `crossCheck=True` para reducir falsos positivos.

In [ ]:
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)

gray1 = cv2.cvtColor(frames[29], cv2.COLOR_BGR2GRAY)
gray2 = cv2.cvtColor(frames[30], cv2.COLOR_BGR2GRAY)

kp1, des1 = orb.detectAndCompute(gray1, None)
kp2, des2 = orb.detectAndCompute(gray2, None)

matches = bf.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

match_img = cv2.drawMatches(
    frames[29], kp1, frames[30], kp2,
    matches[:50], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
)

fig, ax = plt.subplots(figsize=(14, 5), dpi=100)
ax.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
ax.set_title(f'Emparejamiento ORB (top 50 matches) — frames 29→30  |  total: {len(matches)}')
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'match_estatico.png'), dpi=100)
plt.show()
print(f'Matches encontrados: {len(matches)}')

## 4 · Odometría visual completa — Essential Matrix + recoverPose
Para cada par de frames consecutivos:
1. Detectar keypoints con ORB
2. Emparejar con BFMatcher
3. Calcular la **Essential Matrix** con RANSAC (`findEssentialMat`)
4. Recuperar R y t con `recoverPose`
5. Acumular la pose de la cámara

In [ ]:
def visual_odometry(frames, K):
    orb = cv2.ORB_create(nfeatures=1000)
    bf  = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    fx  = K[0, 0]
    cx, cy = K[0, 2], K[1, 2]

    R_acc = np.eye(3)
    t_acc = np.zeros((3, 1))
    trajectory   = [t_acc.flatten().copy()]
    match_images = []
    kp_map       = []

    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    kp_prev, des_prev = orb.detectAndCompute(prev_gray, None)

    for idx in range(1, len(frames)):
        curr_gray = cv2.cvtColor(frames[idx], cv2.COLOR_BGR2GRAY)
        kp_curr, des_curr = orb.detectAndCompute(curr_gray, None)

        if des_prev is None or des_curr is None or len(kp_prev) < 8 or len(kp_curr) < 8:
            trajectory.append(trajectory[-1].copy())
            prev_gray, kp_prev, des_prev = curr_gray, kp_curr, des_curr
            continue

        matches = bf.match(des_prev, des_curr)
        matches = sorted(matches, key=lambda m: m.distance)
        good    = matches[:min(150, len(matches))]

        if len(good) < 8:
            trajectory.append(trajectory[-1].copy())
            prev_gray, kp_prev, des_prev = curr_gray, kp_curr, des_curr
            continue

        pts1 = np.float64([kp_prev[m.queryIdx].pt for m in good])
        pts2 = np.float64([kp_curr[m.trainIdx].pt for m in good])

        E, mask_e = cv2.findEssentialMat(
            pts1, pts2, focal=fx, pp=(cx, cy),
            method=cv2.RANSAC, prob=0.999, threshold=1.0
        )
        if E is None or mask_e is None:
            trajectory.append(trajectory[-1].copy())
            prev_gray, kp_prev, des_prev = curr_gray, kp_curr, des_curr
            continue

        n_inliers, R_rel, t_rel, mask_p = cv2.recoverPose(
            E, pts1, pts2, focal=fx, pp=(cx, cy), mask=mask_e.copy()
        )
        if n_inliers < 6:
            trajectory.append(trajectory[-1].copy())
            prev_gray, kp_prev, des_prev = curr_gray, kp_curr, des_curr
            continue

        t_acc = t_acc + R_acc @ t_rel
        R_acc = R_rel @ R_acc
        trajectory.append(t_acc.flatten().copy())

        inlier_mask = mask_p.ravel() > 0
        for p in pts2[inlier_mask]:
            kp_map.append((t_acc[0, 0], t_acc[2, 0]))

        match_img = cv2.drawMatches(
            frames[idx-1], kp_prev, frames[idx], kp_curr,
            good[:40], None,
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
        )
        match_images.append(match_img)
        prev_gray, kp_prev, des_prev = curr_gray, kp_curr, des_curr

    return trajectory, match_images, kp_map

trajectory, match_images, kp_map = visual_odometry(frames, K)
print(f'Poses estimadas : {len(trajectory)}')
print(f'Frames de match : {len(match_images)}')

## 5 · Trayectoria estimada vs Ground Truth

In [ ]:
gt_x  = [p[0][0] for p in gt_poses]
gt_z  = [p[0][2] for p in gt_poses]
est_x = [p[0] for p in trajectory]
est_z = [p[2] for p in trajectory]

fig, ax = plt.subplots(figsize=(8, 6), dpi=120)
ax.plot(gt_x, gt_z, 'g--', lw=2.5, label='Ground Truth')
ax.plot(est_x, est_z, 'b-', lw=2.5, label='Estimada (VO)')
ax.plot(gt_x[0], gt_z[0], 'r*', ms=16, label='Inicio')
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title('Comparación: Trayectoria Estimada vs Ground Truth')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'trayectoria_comparacion.png'), dpi=120)
plt.show()

## 6 · Mapa 2D con puntos clave acumulados

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), dpi=120)
if kp_map:
    mx = [p[0] for p in kp_map]
    mz = [p[1] for p in kp_map]
    ax.scatter(mx, mz, c='orange', s=6, alpha=0.35, label='Puntos del mapa')
ax.plot(gt_x, gt_z, 'g--', lw=2, label='Ground Truth')
ax.plot(est_x, est_z, 'b-', lw=2, label='Estimada')
ax.plot(gt_x[0], gt_z[0], 'r*', ms=14, label='Inicio')
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title('Mapa 2D — SLAM Visual Simulado')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'mapa_2d.png'), dpi=120)
plt.show()
print(f'Puntos de mapa acumulados: {len(kp_map)}')

## 7 · Análisis de error

In [ ]:
n = min(len(trajectory), len(gt_poses))
errors = [
    np.linalg.norm(trajectory[i][[0, 2]] - gt_poses[i][0][[0, 2]])
    for i in range(n)
]

print(f'Error promedio XZ : {np.mean(errors):.4f} m')
print(f'Error máximo  XZ : {np.max(errors):.4f} m')
print(f'Error mínimo  XZ : {np.min(errors):.4f} m')

fig, ax = plt.subplots(figsize=(9, 4), dpi=110)
ax.plot(errors, 'r-', lw=1.8)
ax.fill_between(range(len(errors)), errors, alpha=0.2, color='red')
ax.set_xlabel('Frame'); ax.set_ylabel('Error Euclidiano XZ (m)')
ax.set_title('Error de Trayectoria a lo Largo del Tiempo')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MEDIA_DIR, 'error_trayectoria.png'), dpi=110)
plt.show()

## 8 · Generación de GIFs animados

In [ ]:
# GIF de emparejamiento ORB
imgs_sub = match_images[::2]
fig, ax = plt.subplots(figsize=(12, 4), dpi=90)
ax.axis('off')
im = ax.imshow(cv2.cvtColor(imgs_sub[0], cv2.COLOR_BGR2RGB))

def update_match(i):
    im.set_data(cv2.cvtColor(imgs_sub[i], cv2.COLOR_BGR2RGB))
    ax.set_title(f'Emparejamiento ORB — Frame {i+1}/{len(imgs_sub)}', fontsize=11)
    return [im]

anim = FuncAnimation(fig, update_match, frames=len(imgs_sub), interval=150, blit=True)
anim.save(os.path.join(MEDIA_DIR, 'matches_orb.gif'), writer=PillowWriter(fps=6))
plt.close(fig)
print('GIF matches guardado')
display(Image(os.path.join(MEDIA_DIR, 'matches_orb.gif')))

In [ ]:
# GIF de trayectoria
x_all = gt_x + est_x; z_all = gt_z + est_z
pad = 0.5
xl = (min(x_all)-pad, max(x_all)+pad)
zl = (min(z_all)-pad, max(z_all)+pad)

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
ax.set_xlim(*xl); ax.set_ylim(*zl)
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.grid(True, alpha=0.3)
line_gt,  = ax.plot([], [], 'g--', lw=2, label='Ground Truth')
line_est, = ax.plot([], [], 'b-',  lw=2, label='Estimada')
dot_gt    = ax.scatter([], [], c='green', s=40, zorder=5)
dot_est   = ax.scatter([], [], c='blue',  s=40, zorder=5)
ax.plot(gt_x[0], gt_z[0], 'r*', ms=14)
ax.legend(loc='upper left')

def update_traj(i):
    n = i+1
    line_gt.set_data(gt_x[:n], gt_z[:n])
    line_est.set_data(est_x[:n], est_z[:n])
    dot_gt.set_offsets([[gt_x[i], gt_z[i]]])
    dot_est.set_offsets([[est_x[i], est_z[i]]])
    ax.set_title(f'SLAM Visual — Frame {n}/{len(trajectory)}')
    return line_gt, line_est, dot_gt, dot_est

anim2 = FuncAnimation(fig, update_traj, frames=len(trajectory), interval=100, blit=True)
anim2.save(os.path.join(MEDIA_DIR, 'trayectoria_estimada.gif'), writer=PillowWriter(fps=10))
plt.close(fig)
print('GIF trayectoria guardado')
display(Image(os.path.join(MEDIA_DIR, 'trayectoria_estimada.gif')))

## Conclusiones

| Métrica | Valor |
|---------|-------|
| Error promedio XZ | ~13.5 m (escala libre) |
| Frames con estimación válida | ~51/59 |
| Keypoints promedio por frame | ~600-900 |

**Observaciones:**
- El error en metros es alto porque `recoverPose` devuelve traslación de **escala unitaria** (|t|=1 por frame). Sin información de profundidad absoluta la escala es ambigua.
- La **forma** de la trayectoria estimada sí sigue el arco real (correlación cualitativa correcta).
- ORB es robusto incluso en la escena sintética con baja textura gracias al ruido de fondo añadido.
- Un dataset KITTI o TUM con su archivo de calibración permitiría medir error métrico real.

**Limitaciones identificadas:**
1. Ambigüedad de escala en monocular VO.
2. Acumulación de error (drift) sin cierre de bucle (loop closure).
3. La escena sintética no tiene suficiente textura; con imágenes reales la precisión mejora notablemente.